# SilicoJev training (Hugging Face data + resumable fine-tuning)

This new notebook leaves silicojev.ipynb unchanged. It fine-tunes Laya's public typed-decision checkpoint for RTL/chip-design and DV decisions: next action, root cause, evidence sufficiency, risk, and urgency. It is a decision model, not free-form chat.

Fresh clone in a terminal: git clone https://github.com/Yaswanth-ampolu/silicojev.git && cd silicojev && jupyter lab silicojev_training_hf.ipynb

Paths are rooted from the cloned repository. Data download and full training are reproducible; smoke training runs separately; full training and HF model upload are gated off by default. No token is embedded in notebook source.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'training' / 'train_silicojev.py').is_file() and (candidate / 'dataset').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from inside a cloned SilicoJev repository.')

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / 'dataset'
BUCKET_DIR = DATA_DIR / 'cache' / 'hf_bucket'
PREPARED_DIR = DATA_DIR / 'prepared' / 'hf_bucket_v1'
MODEL_DIR = REPO_ROOT / 'models' / 'laya-typed-decisions'
LAYA_DIR = REPO_ROOT / 'external' / 'laya'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints' / 'hf_bucket_v1'
EVAL_DIR = REPO_ROOT / 'evaluation' / 'hf_bucket_v1'
for folder in (BUCKET_DIR, MODEL_DIR.parent, LAYA_DIR.parent, CHECKPOINT_DIR, EVAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

HF_BUCKET_ID = 'Yaswanth-ampolu/silicojev'
BUCKET_FILES = ['silicojev_5q.jsonl', 'fixbench_rtl_5q.jsonl', 'rtl_benchls_5q.jsonl', 'veribugbench_5q.jsonl']
BASE_MODEL_ID = 'convaiinnovations/laya-typed-decisions'
BASE_MODEL_REVISION = '1a793eb568e6718f15941d08f85432581df534e3'
LAYA_REPO_URL = 'https://github.com/NandhaKishorM/laya.git'
LAYA_COMMIT = '573e5b62696ba441230cd6be71d593331b5d23af'
SEED, EPOCHS, MAX_LEN, HEAD_MAX_LEN, CHECKPOINT_EVERY = 42, 4, 1024, 256, 100
ENCODER_LR, HEAD_LR, WEIGHT_DECAY = 2.5e-5, 1.0e-4, 0.01
print('Repository:', REPO_ROOT)
print('Repo-relative data/model/checkpoint paths:', BUCKET_DIR.relative_to(REPO_ROOT), MODEL_DIR.relative_to(REPO_ROOT), CHECKPOINT_DIR.relative_to(REPO_ROOT))

## Dependencies and Hugging Face login

The dependency file intentionally does not install or replace PyTorch, preserving the environment's CUDA build. Login uses Hugging Face's interactive secure token prompt; do not paste a token into a code cell. The credential is cached outside the repository. Public data downloads can work anonymously, but account login is needed for upload.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'training' / 'requirements-notebook.txt')], check=True)
from huggingface_hub import HfApi, get_token, login
if not get_token():
    login()  # Secure prompt; never put the token in notebook code/output.
print('Authenticated HF account:', HfApi().whoami().get('name', '<account>'))

In [ ]:
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. Run in the GPU Jupyter environment.')
props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print('GPU:', props.name, '| count:', torch.cuda.device_count())
print('VRAM total/free GiB:', round(total_bytes / 2**30, 2), round(free_bytes / 2**30, 2))
print('BF16 supported:', torch.cuda.is_bf16_supported())
MICRO_BATCH_SIZE = 32 if free_bytes >= 65 * 2**30 else 16
GRAD_ACCUM_STEPS = 2 if MICRO_BATCH_SIZE == 32 else 4
PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
print('Initial micro-batch / accumulation / precision:', MICRO_BATCH_SIZE, GRAD_ACCUM_STEPS, PRECISION)

In [ ]:
from huggingface_hub import download_bucket_files
REFRESH_BUCKET_DATA = False  # Set True only when intentionally refreshing into a new prepared-run directory.
if REFRESH_BUCKET_DATA or any(not (BUCKET_DIR / name).is_file() for name in BUCKET_FILES):
    download_bucket_files(HF_BUCKET_ID, files=[(name, str(BUCKET_DIR / name)) for name in BUCKET_FILES])
missing = [name for name in BUCKET_FILES if not (BUCKET_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing expected bucket files: {missing}')
for name in BUCKET_FILES:
    p = BUCKET_DIR / name
    print(f'{name}: {p.stat().st_size / 1_000_000:.2f} MB')
print('Downloaded into:', BUCKET_DIR.relative_to(REPO_ROOT))

In [ ]:
import hashlib
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
manifest_path = PREPARED_DIR / 'manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    for name, details in manifest['input_files'].items():
        if sha256_file(BUCKET_DIR / name) != details['sha256']:
            raise RuntimeError(f'{name} changed since this prepared run. Choose a new PREPARED_DIR/checkpoint run name.')
elif PREPARED_DIR.exists() and any(PREPARED_DIR.iterdir()):
    raise FileExistsError(f'{PREPARED_DIR} is incomplete and will not be overwritten; choose a new run directory.')
else:
    subprocess.run([
        sys.executable, str(REPO_ROOT / 'training' / 'prepare_hf_bucket_data.py'),
        '--input-dir', str(BUCKET_DIR),
        '--base-split-dir', str(DATA_DIR / 'normalized' / 'merged_5q'),
        '--fixbench-groups', str(DATA_DIR / 'converted' / 'fixbench_rtl' / 'split_groups.json'),
        '--output-dir', str(PREPARED_DIR), '--seed', str(SEED),
    ], check=True)
    manifest = json.loads(manifest_path.read_text())
print('Split record counts:', manifest['split_records'])
print('Source counts by split:', json.dumps(manifest['source_counts_by_split'], indent=2))
print('Source-flagged records excluded from training/evaluation:', manifest['excluded_record_count'])
print('Input SHA-256 values:', json.dumps(manifest['input_files'], indent=2))

## Trust-aware supervision

The preparation step leaves state, questions, and gold distributions intact and adds per-question training_weights. Stronger validated/benchmark/manual labels get weight 1.0; repair-derived inference gets 0.75; teacher, pseudo, synthetic, and unvalidated repair labels get 0.20–0.25. These are explicit experimental weights, not a claim that weak labels are correct. The policy is visible in training/prepare_hf_bucket_data.py; change it and create a new prepared-run directory if you want a different policy. The trainer applies weights to both its soft cross-entropy and RLCD-style proper-scoring loss.

In [ ]:
if LAYA_DIR.exists() and not (LAYA_DIR / '.git').exists():
    raise RuntimeError(f'{LAYA_DIR} exists but is not a Git checkout; move it aside manually before setup.')
if not (LAYA_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', LAYA_REPO_URL, str(LAYA_DIR)], check=True)
dirty = subprocess.run(['git', '-C', str(LAYA_DIR), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError('Laya checkout has local changes; preserving them. Resolve/relocate before pinning the training dependency.')
subprocess.run(['git', '-C', str(LAYA_DIR), 'fetch', 'origin', LAYA_COMMIT], check=True)
current_laya_commit = subprocess.run(['git', '-C', str(LAYA_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if current_laya_commit != LAYA_COMMIT:
    subprocess.run(['git', '-C', str(LAYA_DIR), 'checkout', '--detach', LAYA_COMMIT], check=True)
os.environ['SILICOJEV_LAYA_SOURCE'] = str(LAYA_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(LAYA_DIR)], check=True)
from huggingface_hub import snapshot_download
snapshot_download(repo_id=BASE_MODEL_ID, revision=BASE_MODEL_REVISION, local_dir=str(MODEL_DIR))
required = [MODEL_DIR / 'model.safetensors', MODEL_DIR / 'rl_agent_config.json', MODEL_DIR / 'encoder' / 'config.json']
missing = [str(p.relative_to(REPO_ROOT)) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(f'Base checkpoint incomplete: {missing}')
print('Pinned Laya source and public base checkpoint ready.')

## Smoke test and full run

The smoke test uses a tiny subset, performs one optimizer step, and saves a separate checkpoint. Inspect loss and memory use before setting RUN_FULL_TRAINING = True. H100 starts at BF16, micro-batch 32, accumulation 2; lower the micro-batch if the smoke test OOMs. Full training resumes from its run-local latest checkpoint and refuses incompatible dataset fingerprints.

In [ ]:
SMOKE_DIR = REPO_ROOT / 'checkpoints' / 'smoke_hf_bucket_v1'
SMOKE_DATA = PREPARED_DIR / 'smoke'
if (SMOKE_DIR / 'final' / 'model.safetensors').is_file() and (SMOKE_DIR / 'final' / 'training_state.pt').is_file():
    print('Existing completed smoke checkpoint found; preserving it.')
else:
    if SMOKE_DIR.exists():
        raise FileExistsError(f'{SMOKE_DIR} is incomplete; choose a new smoke run name to preserve it.')
    SMOKE_DATA.mkdir(parents=True, exist_ok=True)
    for split in ('train', 'validation'):
        lines = [line for line in (PREPARED_DIR / f'{split}.jsonl').read_text().splitlines() if line.strip()]
        if len(lines) < 8:
            raise ValueError(f'Not enough {split} rows for smoke test: {len(lines)}')
        (SMOKE_DATA / f'{split}.jsonl').write_text('\n'.join(lines[:8]) + '\n')
    subprocess.run([
        sys.executable, str(REPO_ROOT / 'training' / 'train_silicojev.py'),
        '--model-dir', str(MODEL_DIR), '--train', str(SMOKE_DATA / 'train.jsonl'),
        '--validation', str(SMOKE_DATA / 'validation.jsonl'), '--output-dir', str(SMOKE_DIR),
        '--epochs', '1', '--max-steps', '1', '--micro-batch-size', '2', '--grad-accum-steps', '1',
        '--checkpoint-every', '1', '--resume', 'none', '--dtype', PRECISION, '--seed', str(SEED),
        '--encoder-lr', str(ENCODER_LR), '--head-lr', str(HEAD_LR), '--weight-decay', str(WEIGHT_DECAY),
        '--max-len', str(MAX_LEN), '--head-max-len', str(HEAD_MAX_LEN),
    ], check=True, env=os.environ.copy())
    assert (SMOKE_DIR / 'final' / 'model.safetensors').is_file()
    assert (SMOKE_DIR / 'final' / 'training_state.pt').is_file()
    print('Smoke optimizer step and checkpoint save passed.')

In [ ]:
RUN_FULL_TRAINING = False  # Explicitly enable only after reviewing smoke results.
if (CHECKPOINT_DIR / 'final' / 'model.safetensors').is_file():
    print('A completed final checkpoint already exists; preserving it. Use a new CHECKPOINT_DIR for another training run.')
elif RUN_FULL_TRAINING:
    subprocess.run([
        sys.executable, str(REPO_ROOT / 'training' / 'train_silicojev.py'),
        '--model-dir', str(MODEL_DIR), '--train', str(PREPARED_DIR / 'train.jsonl'),
        '--validation', str(PREPARED_DIR / 'validation.jsonl'), '--output-dir', str(CHECKPOINT_DIR),
        '--epochs', str(EPOCHS), '--micro-batch-size', str(MICRO_BATCH_SIZE),
        '--grad-accum-steps', str(GRAD_ACCUM_STEPS), '--checkpoint-every', str(CHECKPOINT_EVERY),
        '--resume', 'auto', '--dtype', PRECISION, '--seed', str(SEED),
        '--encoder-lr', str(ENCODER_LR), '--head-lr', str(HEAD_LR), '--weight-decay', str(WEIGHT_DECAY),
        '--max-len', str(MAX_LEN), '--head-max-len', str(HEAD_MAX_LEN),
    ], check=True, env=os.environ.copy())
else:
    print('Full training is disabled. Set RUN_FULL_TRAINING=True to start/resume.')

In [ ]:
FINAL_DIR = CHECKPOINT_DIR / 'final'
if FINAL_DIR.exists():
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, str(REPO_ROOT / 'training' / 'evaluate_silicojev.py'),
        '--model-dir', str(FINAL_DIR), '--data', str(PREPARED_DIR / 'test.jsonl'),
        '--output', str(EVAL_DIR / 'test.json'), '--device', 'cuda',
    ], check=True, env=os.environ.copy())
    print((EVAL_DIR / 'test.json').read_text()[:12000])
else:
    print('No full-run final checkpoint found; held-out evaluation skipped.')

## Optional: upload the trained model to Hugging Face

Upload is disabled by default and requires a write-enabled HF token. The default destination is private. Only model/config/tokenizer artifacts are uploaded; optimizer and RNG state (training_state.pt) are excluded.

In [ ]:
UPLOAD_MODEL_TO_HF = False
MODEL_REPO_ID = 'Yaswanth-ampolu/silicojev-model'
MODEL_REPO_PRIVATE = True
if UPLOAD_MODEL_TO_HF:
    from huggingface_hub import HfApi
    import shutil
    if not (FINAL_DIR / 'model.safetensors').is_file():
        raise FileNotFoundError('No final inference checkpoint to upload. Complete full training first.')
    if not (EVAL_DIR / 'test.json').is_file():
        raise FileNotFoundError('Run held-out test evaluation before publishing the model.')
    if not get_token():
        login()
    api = HfApi()
    api.create_repo(repo_id=MODEL_REPO_ID, repo_type='model', private=MODEL_REPO_PRIVATE, exist_ok=True)
    shutil.copyfile(PREPARED_DIR / 'manifest.json', FINAL_DIR / 'training_data_manifest.json')
    evaluation = json.loads((EVAL_DIR / 'test.json').read_text())
    summary = {key: evaluation[key] for key in ('cases', 'decisions', 'model_config', 'metrics', 'by_question_type', 'by_label_source', 'per_source', 'score') if key in evaluation}
    (FINAL_DIR / 'evaluation_summary.json').write_text(json.dumps(summary, indent=2))
    card = ('# SilicoJev\n\nFine-tuned from ' + BASE_MODEL_ID + ' for RTL/chip-design and DV typed decisions.\n\n'
            'Predicts distributions for next action, root cause, evidence sufficiency, risk, and urgency. '
            'Some supervision is teacher-derived or pseudo/unverified; score judgments are exploratory. '
            'See training_data_manifest.json and evaluation_summary.json for dataset hashes and held-out metrics.\n')
    (FINAL_DIR / 'README.md').write_text(card)
    api.upload_folder(
        repo_id=MODEL_REPO_ID, repo_type='model', folder_path=str(FINAL_DIR),
        allow_patterns=['model.safetensors', 'rl_agent_config.json', 'encoder/**', 'tokenizer/**', 'README.md', 'training_data_manifest.json', 'evaluation_summary.json'],
        ignore_patterns=['training_state.pt'],
        commit_message='Upload SilicoJev inference checkpoint',
    )
    print('Uploaded to https://huggingface.co/' + MODEL_REPO_ID)
else:
    print('Upload is disabled; set UPLOAD_MODEL_TO_HF=True after reviewing evaluation.')